In [ ]:
#Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

SEED = 13
N_SPLITS = 5

DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/TFM/DataSet")
INPUT_PATH = DATA_DIR / "05_text_speech_eeg.csv"
PARTITIONS_PATH = DATA_DIR / "data_partitions_paper_ready.csv"

In [ ]:
def load_partitions():
    """Carga directamente las particiones del paper."""
    partitions = pd.read_csv(PARTITIONS_PATH)
    return partitions[["subject_id", "avatar", "outer_fold"]].copy()


def get_metrics(y_true, y_pred, y_score):
    
    return {
        "WAcc": accuracy_score(y_true, y_pred),
        "UAcc": balanced_accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_score),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


def subject_level_predictions(pred_conv):
    # Agrega el margen SVM por sujeto mediante la media.
    pred_subject = (
        pred_conv
        .groupby(["subject_id", "label", "outer_fold"], as_index=False)["score_1"]
        .mean()
    )
    pred_subject["pred"] = (pred_subject["score_1"] >= 0).astype(int)
    return pred_subject

In [ ]:
data = pd.read_csv(INPUT_PATH)
partitions = load_partitions()

text_cols = sorted([c for c in data.columns if c.startswith("text_")], key=lambda c: int(c.split("_", 1)[1]))
speech_cols = sorted([c for c in data.columns if c.startswith("speech_")], key=lambda c: int(c.split("_", 1)[1]))
meta_cols = {"subject_id", "avatar", "label"}
eeg_cols = [c for c in data.columns if c not in meta_cols and c not in text_cols and c not in speech_cols]
feature_cols = text_cols + speech_cols + eeg_cols

required = {"subject_id", "avatar", "label"}
if not required.issubset(data.columns):
    raise ValueError(f"Faltan columnas obligatorias: {required - set(data.columns)}")


df = data.merge(partitions, on=["subject_id", "avatar"], how="inner")

n_before = len(df)
df = df.dropna(subset=feature_cols).copy()
n_removed = n_before - len(df)

print("Filas iniciales con partición:", n_before)
print("Filas eliminadas por no tener alguna modalidad:", n_removed)
print("Filas trimodales finales:", len(df))
print("Sujetos finales:", df["subject_id"].nunique())
print("Variables text:", len(text_cols))
print("Variables speech:", len(speech_cols))
print("Variables EEG:", len(eeg_cols))

print("\nSujetos por outer fold después del filtro:")
display(df.groupby("outer_fold")["subject_id"].nunique().to_frame("n_subjects"))

print("\nDistribución de clases por outer fold:")
display(pd.crosstab(df.drop_duplicates("subject_id")["outer_fold"], df.drop_duplicates("subject_id")["label"]))

print("\nConversaciones disponibles por narrativa:")
display(df["avatar"].value_counts().rename_axis("avatar").to_frame("n_rows"))

Filas iniciales con partición: 600
Filas eliminadas por no tener alguna modalidad: 42
Filas trimodales finales: 558
Sujetos finales: 94
Variables text: 768
Variables speech: 1024
Variables EEG: 27

Sujetos por outer fold después del filtro:


,n_subjects
outer_fold,
1,20
2,18
3,18
4,19
5,19



Distribución de clases por outer fold:


label,0,1
outer_fold,,
1,11,9
2,10,8
3,11,7
4,12,7
5,11,8



Conversaciones disponibles por narrativa:


,n_rows
avatar,
Sad,94
Neutral1,94
Happy,94
Angry,92
Relax,92
Neutral2,92


In [ ]:
# SVM necesita escalado. El escalado se ajusta dentro de cada train/dev, nunca con test.
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(class_weight="balanced", random_state=SEED, cache_size=2000)),
])


param_grid = [
    {"svm__kernel": ["linear"], "svm__C": [0.001, 0.01, 0.1, 1, 10, 100]},
    {"svm__kernel": ["rbf"], "svm__C": [0.01, 0.1, 1, 10, 100], "svm__gamma": ["scale", 0.001, 0.01, 0.1]},
]

n_candidates = 6 + 5 * 4
print("Candidatos SVM por búsqueda:", n_candidates)
print("Fits por búsqueda:", n_candidates * N_SPLITS)

scoring = {
    "WAcc": "accuracy",
    "UAcc": "balanced_accuracy",
    "auc": "roc_auc",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}

metric_cols = ["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]

Candidatos SVM por búsqueda: 26
Fits por búsqueda: 130


In [ ]:
OUT_DIR = DATA_DIR / "05_results_text_speech_eeg_conversation_level_svm"
OUT_DIR.mkdir(parents=True, exist_ok=True)

conv_metrics_rows = []
subject_metrics_rows = []
best_params_rows = []
all_conv_predictions = []

for fold in sorted(df["outer_fold"].unique()):
    print(f"\n===== OUTER FOLD {fold} =====")
    t0 = time.time()

    dev = df[df["outer_fold"] != fold].reset_index(drop=True)
    test = df[df["outer_fold"] == fold].reset_index(drop=True)

    X_dev = dev[feature_cols].to_numpy(dtype=np.float32)
    y_dev = dev["label"].to_numpy(dtype=int)
    groups_dev = dev["subject_id"].to_numpy()

    X_test = test[feature_cols].to_numpy(dtype=np.float32)
    y_test = test["label"].to_numpy(dtype=int)

    inner_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    grid = GridSearchCV(
        estimator=svm_pipeline,
        param_grid=param_grid,
        scoring=scoring,
        refit="UAcc",
        cv=inner_cv,
        n_jobs=-1,
        verbose=0,
    )
    grid.fit(X_dev, y_dev, groups=groups_dev)

    best_idx = grid.best_index_
    model = grid.best_estimator_

    score_1 = model.decision_function(X_test)
    pred = model.predict(X_test)

    pred_conv = test[["subject_id", "avatar", "label", "outer_fold"]].copy()
    pred_conv["score_1"] = score_1
    pred_conv["pred"] = pred
    all_conv_predictions.append(pred_conv)

    conv_metrics = get_metrics(y_test, pred, score_1)
    conv_metrics["outer_fold"] = fold
    conv_metrics["cv_f1"] = grid.cv_results_["mean_test_f1"][best_idx]
    conv_metrics_rows.append(conv_metrics)

    pred_subject = subject_level_predictions(pred_conv)
    subject_metrics = get_metrics(pred_subject["label"], pred_subject["pred"], pred_subject["score_1"])
    subject_metrics["outer_fold"] = fold
    subject_metrics_rows.append(subject_metrics)

    best_params_rows.append({
        "outer_fold": fold,
        "best_kernel": grid.best_params_["svm__kernel"],
        "best_C": grid.best_params_["svm__C"],
        "best_gamma": grid.best_params_.get("svm__gamma", "not_used"),
        "best_inner_UAcc": grid.cv_results_["mean_test_UAcc"][best_idx],
        "best_inner_f1": grid.cv_results_["mean_test_f1"][best_idx],
        "elapsed_seconds": round(time.time() - t0, 1),
    })

    print("Best params:", grid.best_params_)
    print("Conversation-level CV F1:", round(grid.cv_results_["mean_test_f1"][best_idx], 3))
    print("Conversation-level Test F1:", round(conv_metrics["f1"], 3))
    print("Subject-level Test F1:", round(subject_metrics["f1"], 3))
    print("Tiempo fold (s):", round(time.time() - t0, 1))

conv_metrics_df = pd.DataFrame(conv_metrics_rows)
subject_metrics_df = pd.DataFrame(subject_metrics_rows)
best_params_df = pd.DataFrame(best_params_rows)
conv_predictions_df = pd.concat(all_conv_predictions, ignore_index=True)
subject_predictions_global = subject_level_predictions(conv_predictions_df)
global_subject_metrics = get_metrics(
    subject_predictions_global["label"],
    subject_predictions_global["pred"],
    subject_predictions_global["score_1"],
)

results_summary = pd.DataFrame({
    "metric": ["Conversation-level CV F1", "Conversation-level Test F1", "Subject-level Test F1"],
    "mean": [conv_metrics_df["cv_f1"].mean(), conv_metrics_df["f1"].mean(), subject_metrics_df["f1"].mean()],
    "std": [conv_metrics_df["cv_f1"].std(), conv_metrics_df["f1"].std(), subject_metrics_df["f1"].std()],
}).round(3)

conv_metrics_df.to_csv(OUT_DIR / "conversation_level_outer_metrics.csv", index=False)
subject_metrics_df.to_csv(OUT_DIR / "subject_level_outer_metrics.csv", index=False)
best_params_df.to_csv(OUT_DIR / "best_params_by_outer_fold.csv", index=False)
conv_predictions_df.to_csv(OUT_DIR / "conversation_predictions.csv", index=False)
subject_predictions_global.to_csv(OUT_DIR / "subject_predictions_global.csv", index=False)
results_summary.to_csv(OUT_DIR / "main_results_summary.csv", index=False)

print("\nResultados principales")
display(results_summary)

print("\nMétricas subject-level globales")
display(pd.Series(global_subject_metrics).round(3).to_frame("global"))

print("\nArchivos guardados en:", OUT_DIR)


===== OUTER FOLD 1 =====
Best params: {'svm__C': 1, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Conversation-level CV F1: 0.524
Conversation-level Test F1: 0.635
Subject-level Test F1: 0.737
Tiempo fold (s): 18.1

===== OUTER FOLD 2 =====
Best params: {'svm__C': 0.001, 'svm__kernel': 'linear'}
Conversation-level CV F1: 0.563
Conversation-level Test F1: 0.437
Subject-level Test F1: 0.533
Tiempo fold (s): 15.6

===== OUTER FOLD 3 =====
Best params: {'svm__C': 0.1, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Conversation-level CV F1: 0.552
Conversation-level Test F1: 0.476
Subject-level Test F1: 0.462
Tiempo fold (s): 16.9

===== OUTER FOLD 4 =====
Best params: {'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Conversation-level CV F1: 0.496
Conversation-level Test F1: 0.644
Subject-level Test F1: 0.857
Tiempo fold (s): 15.4

===== OUTER FOLD 5 =====
Best params: {'svm__C': 0.001, 'svm__kernel': 'linear'}
Conversation-level CV F1: 0.496
Conversation-level Test F1: 0.667
Subje

,metric,mean,std
0,Conversation-level CV F1,0.526,0.031
1,Conversation-level Test F1,0.572,0.107
2,Subject-level Test F1,0.668,0.164



Métricas subject-level globales


,global
WAcc,0.734
UAcc,0.724
auc,0.730
f1,0.675
precision,0.684
recall,0.667
kappa,0.450



Archivos guardados en: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/05_results_text_speech_eeg_conversation_level_svm_corrected
